# Notebook 07: Global Land Cover Model — RESISC-45

**Runs on:** Google Colab (T4 GPU)

**Runtime:** ~1 hour on T4

**Why:** EuroSAT only has European Sentinel-2 images. This notebook retrains our ResNet-50+SE model on **RESISC-45** — 31,500 Google Earth images from locations **worldwide**, across 45 land use classes mapped down to 10 agriculture-relevant categories.

This makes the model work on **any satellite/Google Earth image from anywhere in the world**.

---
## Instructions
1. Upload to Colab → Runtime → T4 GPU
2. Run all cells
3. Download `global_model_results.zip`
4. Replace model weights in `Phase2_DL/experiments/results/models/`

In [ ]:
!pip install -q torch torchvision timm scikit-learn tqdm gdown

In [ ]:
import os, time, copy, json, warnings, zipfile, shutil
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import timm
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs('results/models', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

## 1. Download RESISC-45 Dataset

**RESISC-45** (Cheng et al., 2017): 31,500 images, 45 classes, 700 images per class.
Images are Google Earth captures from **worldwide locations** at various resolutions (256x256 pixels).

In [ ]:
# Download RESISC-45 from Google Drive (user's uploaded zip)
import gdown, zipfile, os

DATASET_DIR = 'NWPU-RESISC45'

if not os.path.exists(DATASET_DIR):
    ZIP_FILE = 'NWPU-RESISC45.zip'
    
    if not os.path.exists(ZIP_FILE):
        print("Downloading RESISC-45 from Google Drive...")
        gdown.download(
            'https://drive.google.com/uc?id=1VDs8x1RyApEnihchc0OKF_B_Gay-IN5A',
            ZIP_FILE, quiet=False, fuzzy=True
        )
    
    size = os.path.getsize(ZIP_FILE) / 1e6
    print(f"File size: {size:.0f} MB")
    
    print("Extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall('.')
    print("Extraction complete!")
    
    # Find the actual dataset folder (handles nesting)
    for item in os.listdir('.'):
        if os.path.isdir(item) and 'NWPU' in item.upper():
            DATASET_DIR = item
            break
    for item in os.listdir('.'):
        if os.path.isdir(item):
            sub = os.path.join(item, 'NWPU-RESISC45')
            if os.path.isdir(sub):
                DATASET_DIR = sub
                break
else:
    print(f"Dataset already exists at {DATASET_DIR}")

# Verify
folders = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f"\nDataset: {DATASET_DIR}")
print(f"Class folders: {len(folders)}")
print(f"First 15: {folders[:15]}")

# Show samples
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, d in enumerate(folders[:5]):
    img_dir = os.path.join(DATASET_DIR, d)
    img_file = sorted(os.listdir(img_dir))[0]
    img = Image.open(os.path.join(img_dir, img_file))
    n = len(os.listdir(img_dir))
    axes[i].imshow(img)
    axes[i].set_title(f'{d} ({n})', fontsize=9, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('RESISC-45 Preview — Verify these are satellite images', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Map 45 Classes → 10 Agriculture-Relevant Categories

We group the 45 fine-grained classes into 10 meaningful land cover categories for agriculture monitoring.

In [ ]:
# Map 45 classes → 10 agriculture-relevant classes
CLASS_MAPPING = {
    # Farmland (agriculture)
    'circular_farmland': 'Farmland',
    'rectangular_farmland': 'Farmland',
    'terrace': 'Farmland',
    
    # Meadow / Pasture
    'meadow': 'Meadow',
    
    # Forest
    'forest': 'Forest',
    'chaparral': 'Forest',
    
    # Residential
    'dense_residential': 'Residential',
    'medium_residential': 'Residential',
    'sparse_residential': 'Residential',
    'mobile_home_park': 'Residential',
    
    # Industrial / Commercial
    'industrial_area': 'Industrial',
    'commercial_area': 'Industrial',
    'storage_tank': 'Industrial',
    
    # Highway / Roads
    'freeway': 'Highway',
    'overpass': 'Highway',
    'intersection': 'Highway',
    'roundabout': 'Highway',
    'bridge': 'Highway',
    
    # River
    'river': 'River',
    
    # Lake / Water
    'lake': 'Lake',
    'beach': 'Lake',
    'wetland': 'Lake',
    
    # Desert / Barren
    'desert': 'Desert',
    
    # Mountain
    'mountain': 'Mountain',
}

# Target class names (sorted)
TARGET_CLASSES = sorted(set(CLASS_MAPPING.values()))
TARGET_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}

print(f"Mapped to {len(TARGET_CLASSES)} target classes:")
for i, c in enumerate(TARGET_CLASSES):
    source_classes = [k for k, v in CLASS_MAPPING.items() if v == c]
    print(f"  {i}: {c} ← {source_classes}")

In [ ]:
# Build image paths and labels
all_paths = []
all_labels = []
skipped = []

for source_class, target_class in CLASS_MAPPING.items():
    class_dir = os.path.join(DATASET_DIR, source_class)
    if not os.path.exists(class_dir):
        skipped.append(source_class)
        continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.bmp')):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(TARGET_TO_IDX[target_class])

if skipped:
    print(f"Skipped (not found): {skipped}")

print(f"Total images: {len(all_paths)}")
print(f"\nClass distribution:")
label_counts = Counter(all_labels)
for idx in sorted(label_counts.keys()):
    print(f"  {TARGET_CLASSES[idx]:<15} {label_counts[idx]:>5} images")

if len(all_paths) == 0:
    print("\nERROR: No images found!")
    print(f"DATASET_DIR = {DATASET_DIR}")
    print(f"Exists: {os.path.exists(DATASET_DIR)}")
    if os.path.exists(DATASET_DIR):
        contents = sorted(os.listdir(DATASET_DIR))[:20]
        print(f"Contents: {contents}")
    print("\nExpected folder names like: forest, river, desert, meadow, ...")
    print("If folder names differ, we need to update CLASS_MAPPING.")

In [ ]:
# Visualize samples from each mapped class
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, cls_name in enumerate(TARGET_CLASSES):
    cls_label = TARGET_TO_IDX[cls_name]
    # Find first image with this label
    found = False
    for i, lbl in enumerate(all_labels):
        if lbl == cls_label:
            img = Image.open(all_paths[i])
            axes[idx].imshow(img)
            count = label_counts.get(cls_label, 0)
            axes[idx].set_title(f'{cls_name}\n({count} imgs)', fontsize=10, fontweight='bold')
            found = True
            break
    if not found:
        axes[idx].set_title(f'{cls_name}\n(0 imgs)', fontsize=10, color='red')
        axes[idx].text(0.5, 0.5, 'Not found', ha='center', va='center', transform=axes[idx].transAxes)
    axes[idx].axis('off')

plt.suptitle('RESISC-45 → 10 Mapped Classes (Google Earth, Worldwide)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/resisc45_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Dataset & DataLoaders

In [ ]:
# Custom dataset
class RESISC45Dataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# Stratified split: 70/15/15
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

# Transforms - heavier augmentation for global robustness
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=[90, 90])], p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    ], p=0.4),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Datasets & Loaders
BATCH_SIZE = 64
train_dataset = RESISC45Dataset(train_paths, train_labels, train_transform)
val_dataset = RESISC45Dataset(val_paths, val_labels, eval_transform)
test_dataset = RESISC45Dataset(test_paths, test_labels, eval_transform)

# Class weights
train_counts = Counter(train_labels)
class_weights = torch.tensor(
    [len(train_labels) / (len(TARGET_CLASSES) * train_counts[i]) for i in range(len(TARGET_CLASSES))],
    dtype=torch.float32
).to(device)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches - Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 4. Model Architecture (Same ResNet-50+SE)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(True),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.size()[:2]
        y = self.excitation(self.squeeze(x).view(b, c)).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ResNetSE(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        self.se_block = SEBlock(2048, 16)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(2048, 512), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(512, num_classes))

    def forward(self, x):
        f = self.backbone.forward_features(x)
        f = self.se_block(f)
        return self.classifier(F.adaptive_avg_pool2d(f, 1).flatten(1))


NUM_CLASSES = len(TARGET_CLASSES)
model = ResNetSE(num_classes=NUM_CLASSES, pretrained=True).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50+SE: {params:,} parameters, {NUM_CLASSES} classes")
print(f"Classes: {TARGET_CLASSES}")

## 5. Training

In [ ]:
# Training infrastructure
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience, self.counter, self.best_loss, self.should_stop = patience, 0, None, False
    def __call__(self, val_loss):
        if self.best_loss is None: self.best_loss = val_loss
        elif val_loss > self.best_loss - 1e-4:
            self.counter += 1
            if self.counter >= self.patience: self.should_stop = True
        else: self.best_loss, self.counter = val_loss, 0


criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)
early_stopping = EarlyStopping(patience=10)

NUM_EPOCHS = 40
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
best_state = None

print(f"Training ResNet-50+SE on RESISC-45 ({NUM_CLASSES} classes)")
print(f"Epochs: {NUM_EPOCHS} | LR: 1e-4 | Batch: {BATCH_SIZE}")
print("="*60)

t0 = time.time()
for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        correct += out.max(1)[1].eq(labels).sum().item()
        total += labels.size(0)
    train_loss /= total
    train_acc = correct / total

    # Validate
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_loss += criterion(out, labels).item() * imgs.size(0)
            correct += out.max(1)[1].eq(labels).sum().item()
            total += labels.size(0)
    val_loss /= total
    val_acc = correct / total

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - t0
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train: {train_loss:.4f} / {train_acc:.4f} | "
              f"Val: {val_loss:.4f} / {val_acc:.4f} | {elapsed:.0f}s")

    early_stopping(val_loss)
    if early_stopping.should_stop:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
print(f"\nDone in {time.time()-t0:.0f}s | Best val acc: {best_val_acc:.4f}")

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', linewidth=2, color='#22c55e')
axes[0].plot(epochs, history['val_loss'], label='Val', linewidth=2, color='#ef4444')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train', linewidth=2, color='#22c55e')
axes[1].plot(epochs, [a*100 for a in history['val_acc']], label='Val', linewidth=2, color='#ef4444')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('RESISC-45 Global Model — Learning Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/resisc45_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test Set Evaluation

In [ ]:
# Evaluate on test set
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(device)).max(1)[1].cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

acc = accuracy_score(all_true, all_preds)
f1 = f1_score(all_true, all_preds, average='macro')

print(f"Test Accuracy: {acc:.4f} ({acc*100:.1f}%)")
print(f"Test F1-Macro: {f1:.4f}")
print()
print(classification_report(all_true, all_preds, target_names=TARGET_CLASSES, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=ax, vmin=0, vmax=1)
ax.set_title(f'RESISC-45 Global Model — Confusion Matrix\nAccuracy: {acc:.3f}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
plt.tight_layout()
plt.savefig('results/figures/resisc45_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Model & Config

In [ ]:
# Save model weights (same filename so webapp picks it up automatically)
torch.save(model.state_dict(), 'results/models/ResNet50_SE.pth')
print("Model saved: results/models/ResNet50_SE.pth")

# Save class config (webapp needs this)
model_config = {
    'class_names': TARGET_CLASSES,
    'num_classes': NUM_CLASSES,
    'dataset': 'RESISC-45',
    'accuracy': round(acc, 4),
    'f1_macro': round(f1, 4),
    'architecture': 'ResNet-50 + SE Attention',
    'class_mapping': CLASS_MAPPING,
}
with open('results/metrics/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Config saved: results/metrics/model_config.json")

# Save training history
with open('results/metrics/resisc45_history.json', 'w') as f:
    json.dump(history, f)

# Save metrics
import pandas as pd
pd.DataFrame([{
    'Model': 'ResNet50_SE_Global',
    'Dataset': 'RESISC-45',
    'Classes': NUM_CLASSES,
    'Accuracy': f'{acc:.4f}',
    'F1_Macro': f'{f1:.4f}',
}]).to_csv('results/metrics/resisc45_results.csv', index=False)

print("\nAll saved!")

## 8. Download Results

In [ ]:
# List files
print("Generated files:")
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        fp = os.path.join(root, f)
        print(f"  {fp:<50} ({os.path.getsize(fp)/1024:.0f} KB)")

# Zip
shutil.make_archive('global_model_results', 'zip', '.', 'results')
print(f"\nDownload: global_model_results.zip")

try:
    from google.colab import files
    files.download('global_model_results.zip')
except ImportError:
    print("Download global_model_results.zip manually.")

## Summary

**Trained ResNet-50+SE on RESISC-45 (global Google Earth imagery)**

- 31,500 images from worldwide locations
- 45 original classes → 10 agriculture-relevant categories
- **Farmland, Meadow, Forest, Residential, Industrial, Highway, River, Lake, Desert, Mountain**
- Model should now work on Google Earth screenshots from **any country**

**Next steps:**
1. Download `global_model_results.zip`
2. Extract to `Phase2_DL/experiments/results/`
3. Copy `model_config.json` so the webapp picks up new class names
4. Test with Google Earth screenshots from India, Africa, Americas — anywhere!